In [4]:
import os

os.chdir("../")
os.getcwd()

'/cluster/work/krisnol/jormungandr'

In [5]:
import pandas as pd
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
import torch

In [29]:

path = "data/MOT17/"
train_path = path + "train/MOT17-02/"

# Coloums
# frame_number, identity_number, bbox_left, bbox_top, bbox_width, bbox_height, confidence_score, class, visibility

columns = ["frame_number", "identity_number", "bbox_left", "bbox_top", "bbox_width", "bbox_height", "confidence_score", "class", "visibility"]
gt_train = pd.read_csv(train_path + "gt/gt.txt", sep=",", header=None, names=columns)


In [30]:
gt_train

,frame_number,identity_number,bbox_left,bbox_top,bbox_width,bbox_height,confidence_score,class,visibility
0,1,1,912,484,97,109,0,7,1.0
1,2,1,912,484,97,109,0,7,1.0
2,3,1,912,484,97,109,0,7,1.0
3,4,1,912,484,97,109,0,7,1.0
4,5,1,912,484,97,109,0,7,1.0
...,...,...,...,...,...,...,...,...,...
29998,593,80,1043,445,32,97,1,1,0.0
29999,594,80,1043,445,32,97,1,1,0.0
30000,600,81,1007,451,24,69,1,1,0.0
30001,600,82,987,473,21,43,1,1,0.0


In [31]:
filtered_gt_train = gt_train.query("confidence_score != 0")
filtered_gt_train

,frame_number,identity_number,bbox_left,bbox_top,bbox_width,bbox_height,confidence_score,class,visibility
600,1,2,1338,418,167,379,1,1,1.0
601,2,2,1342,417,168,380,1,1,1.0
602,3,2,1346,417,170,380,1,1,1.0
603,4,2,1351,417,171,381,1,1,1.0
604,5,2,1355,417,173,381,1,1,1.0
...,...,...,...,...,...,...,...,...,...
29998,593,80,1043,445,32,97,1,1,0.0
29999,594,80,1043,445,32,97,1,1,0.0
30000,600,81,1007,451,24,69,1,1,0.0
30001,600,82,987,473,21,43,1,1,0.0


In [32]:
filtered_gt_train["bbox"] = filtered_gt_train[["bbox_left", "bbox_top", "bbox_width", "bbox_height"]].values.tolist()
filtered_gt_train = filtered_gt_train[["frame_number", "bbox", "class", "visibility"]]
filtered_gt_train

/tmp/ipykernel_239381/511642379.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_gt_train["bbox"] = filtered_gt_train[["bbox_left", "bbox_top", "bbox_width", "bbox_height"]].values.tolist()


,frame_number,bbox,class,visibility
600,1,"[1338, 418, 167, 379]",1,1.0
601,2,"[1342, 417, 168, 380]",1,1.0
602,3,"[1346, 417, 170, 380]",1,1.0
603,4,"[1351, 417, 171, 381]",1,1.0
604,5,"[1355, 417, 173, 381]",1,1.0
...,...,...,...,...
29998,593,"[1043, 445, 32, 97]",1,0.0
29999,594,"[1043, 445, 32, 97]",1,0.0
30000,600,"[1007, 451, 24, 69]",1,0.0
30001,600,"[987, 473, 21, 43]",1,0.0


In [33]:
combined = filtered_gt_train.groupby("frame_number").agg({"bbox": list, "class": list, "visibility": list}).reset_index()
combined

,frame_number,bbox,class,visibility
0,1,"[[1338, 418, 167, 379], [586, 447, 85, 263], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51351, 0.94595, 1.0, 1.0, 0.98687..."
1,2,"[[1342, 417, 168, 380], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.5163, 0.94643, 1.0, 1.0, 0.98666,..."
2,3,"[[1346, 417, 170, 380], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51366, 0.94643, 1.0, 1.0, 0.98666..."
3,4,"[[1351, 417, 171, 381], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51099, 0.94643, 1.0, 1.0, 0.98645..."
4,5,"[[1355, 417, 173, 381], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.50829, 0.94643, 1.0, 1.0, 0.98645..."
...,...,...,...,...
595,596,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.085047, 0.13523, 0.2027, 1.0, 0.0..."
596,597,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.56505, 0.14369, 0.22491, 1.0, 0.0..."
597,598,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.55093, 0.14098, 0.25149, 1.0, 0.0..."
598,599,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.56644, 0.14927, 0.27751, 1.0, 0.0..."


In [34]:
image_paths = []
for frame_number in combined["frame_number"]:
    image_file = f"{frame_number:06d}.jpg"
    image_path = os.path.join(train_path, "img1", image_file)
    image_paths.append(image_path)

combined["image_path"] = image_paths
combined

,frame_number,bbox,class,visibility,image_path
0,1,"[[1338, 418, 167, 379], [586, 447, 85, 263], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51351, 0.94595, 1.0, 1.0, 0.98687...",data/MOT17/train/MOT17-02/img1/000001.jpg
1,2,"[[1342, 417, 168, 380], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.5163, 0.94643, 1.0, 1.0, 0.98666,...",data/MOT17/train/MOT17-02/img1/000002.jpg
2,3,"[[1346, 417, 170, 380], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51366, 0.94643, 1.0, 1.0, 0.98666...",data/MOT17/train/MOT17-02/img1/000003.jpg
3,4,"[[1351, 417, 171, 381], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.51099, 0.94643, 1.0, 1.0, 0.98645...",data/MOT17/train/MOT17-02/img1/000004.jpg
4,5,"[[1355, 417, 173, 381], [586, 446, 85, 264], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1.0, 1.0, 0.50829, 0.94643, 1.0, 1.0, 0.98645...",data/MOT17/train/MOT17-02/img1/000005.jpg
...,...,...,...,...,...
595,596,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.085047, 0.13523, 0.2027, 1.0, 0.0...",data/MOT17/train/MOT17-02/img1/000596.jpg
596,597,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.56505, 0.14369, 0.22491, 1.0, 0.0...",data/MOT17/train/MOT17-02/img1/000597.jpg
597,598,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.55093, 0.14098, 0.25149, 1.0, 0.0...",data/MOT17/train/MOT17-02/img1/000598.jpg
598,599,"[[1118, 429, 42, 145], [1061, 445, 42, 122], [...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.0, 0.56644, 0.14927, 0.27751, 1.0, 0.0...",data/MOT17/train/MOT17-02/img1/000599.jpg


In [ ]:
image_folder = train_path + "img1/"
image_files = sorted(os.listdir(image_folder))


for image_file
    



SyntaxError: incomplete input (1995274987.py, line 7)